In [1]:
import pandas as pd
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive
from src.analysis.poissonFunctions import (
    compute_bayesian_lambda,
    compute_bayesian_lambda_assists,
    compute_bayesian_lambda_rebounds,
    compute_bayesian_lambda_blocks,
    compute_bayesian_lambda_steals
)

from scipy.stats import poisson
import numpy as np
from nba_api.stats.endpoints import leaguedashteamstats

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
s26.rename(columns={'BLK_x': 'BLK'}, inplace=True)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_DFS_20251208_142406.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,Pascal Siakam,Over,24.5,-137,2025-12-09,2025-12-08T22:23:21Z,2025-12-08 14:24:06
1,Underdog,player_points,Pascal Siakam,Under,24.5,-137,2025-12-09,2025-12-08T22:23:21Z,2025-12-08 14:24:06
2,Underdog,player_points,Zach LaVine,Over,21.5,-137,2025-12-09,2025-12-08T22:23:21Z,2025-12-08 14:24:06
3,Underdog,player_points,Zach LaVine,Under,21.5,-137,2025-12-09,2025-12-08T22:23:21Z,2025-12-08 14:24:06
4,Underdog,player_points,Andrew Nembhard,Over,16.5,-137,2025-12-09,2025-12-08T22:23:21Z,2025-12-08 14:24:06


## Points

### prizepicks

In [3]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_off_rtg, league_avg_def_rtg,
        league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)
    else:
        # Handle other cases
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'L-5': round(count_line_hits(player_df, target_pts, 'player_points', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_pts, 'player_points', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_pts, 'player_points', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })

point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.to_csv(f'data/props/prizepicks/player_points.csv', index=False)
point_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Devin Vassell,15.5,0.6,0.6,0.47,0.902,0.098
1,Jose Alvarado,8.5,0.8,0.6,0.47,0.833,0.167
2,Mark Williams,12.0,0.8,0.6,0.60,0.758,0.242
3,Rudy Gobert,10.5,0.6,0.6,0.60,0.680,0.320
4,Saddiq Bey,16.5,1.0,0.8,0.60,0.666,0.334
5,De'Aaron Fox,24.5,0.8,0.9,0.60,0.655,0.345
6,Stephon Castle,14.5,0.6,0.7,0.60,0.560,0.440
7,Andrew Nembhard,16.5,0.2,0.3,0.40,0.559,0.441
8,Royce O'Neale,9.5,0.4,0.4,0.53,0.557,0.443
9,Anthony Edwards,29.0,0.6,0.6,0.60,0.554,0.446


### underdog

In [4]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_off_rtg, league_avg_def_rtg,
        league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)
    else:
        # Handle other cases
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'L-5': round(count_line_hits(player_df, target_pts, 'player_points', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_pts, 'player_points', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_pts, 'player_points', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })

point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.to_csv(f'data/props/underdog/player_points.csv', index=False)
point_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Jose Alvarado,8.5,0.8,0.6,0.47,0.833,0.167
1,Precious Achiuwa,6.5,0.8,0.7,0.60,0.707,0.293
2,Saddiq Bey,16.5,1.0,0.8,0.60,0.666,0.334
3,Anthony Edwards,28.5,0.6,0.6,0.60,0.626,0.374
4,Andrew Nembhard,16.5,0.2,0.3,0.40,0.559,0.441
5,Royce O'Neale,9.5,0.4,0.4,0.53,0.557,0.443
6,Russell Westbrook,12.5,0.4,0.4,0.47,0.547,0.453
7,Dillon Brooks,22.5,0.6,0.6,0.47,0.536,0.464
8,Jordan Hawkins,6.5,0.6,0.4,0.33,0.519,0.481
9,Jeremiah Fears,16.5,0.6,0.4,0.47,0.517,0.483


## Assists

### prizepicks

In [5]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_assists(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_def_rtg, league_avg_pace,
        league_avg_ast_ratio, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'L-5': round(count_line_hits(player_df, target_ast, 'player_assists', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_ast, 'player_assists', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_ast, 'player_assists', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.to_csv(f'data/props/prizepicks/player_assists.csv', index=False)
assist_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Stephon Castle,5.0,0.8,0.7,0.60,0.902,0.098
1,Russell Westbrook,7.0,0.6,0.4,0.47,0.613,0.387
2,Trey Murphy III,3.5,1.0,0.6,0.60,0.599,0.401
3,Keldon Johnson,1.5,0.6,0.7,0.67,0.543,0.457
4,Julius Randle,5.5,0.8,0.5,0.53,0.525,0.475
5,Jaden McDaniels,2.5,0.6,0.6,0.67,0.509,0.491
6,DeMar DeRozan,3.5,0.4,0.6,0.47,0.497,0.503
7,Derik Queen,4.0,0.6,0.5,0.47,0.471,0.529
8,Isaiah Jackson,0.5,0.2,0.4,0.53,0.461,0.539
9,De'Aaron Fox,6.0,0.4,0.5,0.40,0.444,0.556


### underdog

In [6]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_assists(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_def_rtg, league_avg_pace,
        league_avg_ast_ratio, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'L-5': round(count_line_hits(player_df, target_ast, 'player_assists', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_ast, 'player_assists', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_ast, 'player_assists', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.to_csv(f'data/props/underdog/player_assists.csv', index=False)
assist_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Naz Reid,2.5,0.8,0.5,0.60,0.554,0.446
1,Keldon Johnson,1.5,0.6,0.7,0.67,0.543,0.457
2,Julius Randle,5.5,0.8,0.5,0.53,0.525,0.475
3,Bennedict Mathurin,2.5,0.6,0.4,0.33,0.443,0.557
4,Dylan Harper,3.5,0.6,0.6,0.40,0.369,0.631
5,Grayson Allen,3.5,0.2,0.4,0.60,0.345,0.655
6,Malik Monk,3.5,0.6,0.3,0.27,0.310,0.690
7,Collin Gillespie,5.5,0.2,0.4,0.33,0.282,0.718


# REBOUNDS

### prizepicks

In [7]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_rebounds(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_reb,
        league_avg_oreb, league_avg_dreb, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'L-5': round(count_line_hits(player_df, target_reb, 'player_rebounds', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_reb, 'player_rebounds', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_reb, 'player_rebounds', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.to_csv(f'data/props/prizepicks/player_rebounds.csv', index=False)
rebound_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Jarace Walker,3.5,0.4,0.6,0.60,0.720,0.280
1,Stephon Castle,4.5,0.6,0.8,0.73,0.650,0.350
2,Trey Murphy III,5.5,0.4,0.4,0.53,0.649,0.351
3,Saddiq Bey,6.0,0.4,0.5,0.53,0.628,0.372
4,Devin Vassell,3.5,0.4,0.5,0.40,0.565,0.435
5,Mark Williams,8.5,0.6,0.3,0.33,0.562,0.438
6,Pascal Siakam,7.5,0.6,0.5,0.33,0.546,0.454
7,Rudy Gobert,10.5,0.4,0.6,0.53,0.510,0.490
8,Jeremiah Fears,3.5,0.4,0.5,0.47,0.507,0.493
9,Yves Missi,6.0,0.4,0.4,0.33,0.450,0.550


### underdog

In [8]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_rebounds(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_reb,
        league_avg_oreb, league_avg_dreb, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'L-5': round(count_line_hits(player_df, target_reb, 'player_rebounds', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_reb, 'player_rebounds', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_reb, 'player_rebounds', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.to_csv(f'data/props/underdog/player_rebounds.csv', index=False)
rebound_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Stephon Castle,4.5,0.6,0.8,0.73,0.650,0.350
1,Trey Murphy III,5.5,0.4,0.4,0.53,0.649,0.351
2,Jose Alvarado,2.5,0.6,0.7,0.53,0.595,0.405
3,Harrison Barnes,3.5,0.8,0.7,0.60,0.577,0.423
4,Rudy Gobert,10.5,0.4,0.6,0.53,0.510,0.490
5,Jeremiah Fears,3.5,0.4,0.5,0.47,0.507,0.493
6,Ryan Dunn,4.5,0.2,0.2,0.33,0.386,0.614
7,Bennedict Mathurin,5.5,0.4,0.4,0.33,0.367,0.633
8,DeMar DeRozan,3.5,0.2,0.2,0.20,0.363,0.637
9,Malik Monk,2.5,0.4,0.3,0.33,0.359,0.641


## Blocks

### prizepicks

In [9]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_blocks')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_blk = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_blocks(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_blk % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_blk) + 1)
    elif target_blk % 1 == 0:
        prob_over_poisson = poisson.sf(target_blk, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_blk), int(target_blk) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_blk,
        'L-5': round(count_line_hits(player_df, target_blk, 'player_blocks', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_blk, 'player_blocks', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_blk, 'player_blocks', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
blocks_df = pd.DataFrame(res)
blocks_df.to_csv(f'data/props/prizepicks/player_blocks.csv', index=False)
blocks_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%,IMPLIED_ODDS
0,Maxime Raynaud,0.5,0.8,0.6,0.47,0.497,0.503,2.012
1,Oso Ighodaro,0.5,0.4,0.4,0.40,0.379,0.621,2.637


### underdog

In [10]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_blocks')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_blk = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_blocks(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_blk % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_blk) + 1)
    elif target_blk % 1 == 0:
        prob_over_poisson = poisson.sf(target_blk, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_blk), int(target_blk) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_blk,
        'L-5': round(count_line_hits(player_df, target_blk, 'player_blocks', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_blk, 'player_blocks', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_blk, 'player_blocks', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
blocks_df = pd.DataFrame(res)
blocks_df.to_csv(f'data/props/underdog/player_blocks.csv', index=False)
blocks_df.head(10)

""


# STEALS

### prizepicks

In [11]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_steals')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_stl = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_steals(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_stl % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_stl) + 1)
    elif target_stl % 1 == 0:
        prob_over_poisson = poisson.sf(target_stl, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_stl), int(target_stl) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_stl,
        'L-5': round(count_line_hits(player_df, target_stl, 'player_steals', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_stl, 'player_steals', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_stl, 'player_steals', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
    steals_df = pd.DataFrame(res)
    steals_df.to_csv(f'data/props/prizepicks/player_steals.csv', index=False)
    steals_df.head(10)

### underdog

In [12]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_steals')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_stl = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_steals(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_stl % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_stl) + 1)
    elif target_stl % 1 == 0:
        prob_over_poisson = poisson.sf(target_stl, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_stl), int(target_stl) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_stl,
        'L-5': round(count_line_hits(player_df, target_stl, 'player_steals', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_stl, 'player_steals', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_stl, 'player_steals', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
    steals_df = pd.DataFrame(res)
    steals_df.to_csv(f'data/props/underdog/player_steals.csv', index=False)
    steals_df.head(10)

### prizepicks

In [13]:
## COMBO PROPS - All Categories

combo_categories = [
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_turnovers',
    'player_blocks_steals'
]

for category in combo_categories:
    print(f"\nProcessing {category}...")
    
    dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == category)]
    
    if dfs_data.empty:
        print(f"No data found for {category}")
        continue
    
    res = []
    PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))
    
    for _, row in PLAYERS.iterrows():
        PLAYER = row['NAME']
        target_line = row['LINE']

        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        if player_df.empty:
            continue

        res.append({
            'NAME': PLAYER,
            'LINE': target_line,
            'L-5': count_line_hits(player_df, target_line, category, [5])['L-5'],
            'L-10': count_line_hits(player_df, target_line, category, [10])['L-10'],
            'L-15': count_line_hits(player_df, target_line, category, [15])['L-15'],
        })
    
    if res:
        combo_df = pd.DataFrame(res).sort_values(by='L-5', ascending=False).reset_index(drop=True)
        combo_df.to_csv(f'data/props/prizepicks/{category}.csv', index=False)
        print(f"Saved {len(combo_df)} players for {category}")
        display(combo_df)
    else:
        print(f"No results for {category}")


Processing player_points_rebounds_assists...
Saved 40 players for player_points_rebounds_assists


,NAME,LINE,L-5,L-10,L-15
0,Julian Champagnie,14.5,0.8,0.7,0.67
1,Keldon Johnson,20.5,0.8,0.9,0.67
2,Jay Huff,18.5,0.8,0.5,0.40
3,Devin Vassell,21.5,0.8,0.8,0.53
4,Stephon Castle,24.0,0.8,0.9,0.73
5,Saddiq Bey,25.5,0.8,0.7,0.53
6,Royce O'Neale,18.0,0.6,0.4,0.53
7,Donte DiVincenzo,22.5,0.6,0.4,0.33
8,Jordan Hawkins,9.5,0.6,0.5,0.40
9,Yves Missi,13.5,0.6,0.5,0.47



Processing player_points_rebounds...
Saved 41 players for player_points_rebounds


,NAME,LINE,L-5,L-10,L-15
0,Pascal Siakam,32.5,0.8,0.6,0.47
1,Saddiq Bey,23.0,0.8,0.7,0.53
2,Stephon Castle,18.5,0.8,0.9,0.80
3,Devin Vassell,19.5,0.8,0.7,0.47
4,Jeremiah Fears,20.5,0.8,0.5,0.53
5,Donte DiVincenzo,17.5,0.8,0.6,0.47
6,Maxime Raynaud,19.5,0.6,0.3,0.20
7,Jaden McDaniels,20.5,0.6,0.3,0.40
8,Jordan Hawkins,8.5,0.6,0.5,0.40
9,Jose Alvarado,11.5,0.6,0.5,0.40



Processing player_points_assists...
Saved 35 players for player_points_assists


...

,NAME,LINE,L-5,L-10,L-15
0,Stephon Castle,9.5,0.8,0.8,0.67
1,Harrison Barnes,5.5,0.8,0.7,0.53
2,Naz Reid,9.0,0.8,0.5,0.40
3,Trey Murphy III,9.5,0.8,0.6,0.67
4,Jose Alvarado,5.5,0.6,0.7,0.53
5,Mark Williams,9.5,0.6,0.3,0.33
6,Jeremiah Fears,7.5,0.6,0.5,0.33
7,Donte DiVincenzo,8.0,0.6,0.4,0.33
8,Saddiq Bey,8.0,0.6,0.6,0.53
9,Luke Kornet,9.5,0.6,0.4,0.33



Processing player_turnovers...
Saved 6 players for player_turnovers


,NAME,LINE,L-5,L-10,L-15
0,Jose Alvarado,1.5,0.8,0.7,0.60
1,Jordan Hawkins,0.5,0.6,0.6,0.53
2,Julian Champagnie,0.5,0.6,0.8,0.87
3,Jeremiah Fears,2.5,0.4,0.4,0.53
4,Malik Monk,1.5,0.2,0.1,0.13
5,Donte DiVincenzo,1.5,0.0,0.3,0.33



Processing player_blocks_steals...
Saved 6 players for player_blocks_steals


,NAME,LINE,L-5,L-10,L-15
0,Ryan Dunn,1.5,1.0,0.7,0.67
1,Pascal Siakam,1.5,0.6,0.7,0.53
2,Isaiah Jackson,1.5,0.6,0.7,0.60
3,Collin Gillespie,1.5,0.6,0.6,0.40
4,De'Aaron Fox,1.5,0.4,0.5,0.40
5,Derik Queen,1.5,0.2,0.5,0.53


### underdog

In [14]:
## COMBO PROPS - All Categories

combo_categories = [
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_turnovers',
    'player_blocks_steals'
]

for category in combo_categories:
    print(f"\nProcessing {category}...")
    
    dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == category)]
    
    if dfs_data.empty:
        print(f"No data found for {category}")
        continue
    
    res = []
    PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))
    
    for _, row in PLAYERS.iterrows():
        PLAYER = row['NAME']
        target_line = row['LINE']

        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        if player_df.empty:
            continue

        res.append({
            'NAME': PLAYER,
            'LINE': target_line,
            'L-5': count_line_hits(player_df, target_line, category, [5])['L-5'],
            'L-10': count_line_hits(player_df, target_line, category, [10])['L-10'],
            'L-15': count_line_hits(player_df, target_line, category, [15])['L-15'],
        })
    
    if res:
        combo_df = pd.DataFrame(res).sort_values(by='L-5', ascending=False).reset_index(drop=True)
        combo_df.to_csv(f'data/props/underdog/{category}.csv', index=False)
        print(f"Saved {len(combo_df)} players for {category}")
        display(combo_df)
    else:
        print(f"No results for {category}")


Processing player_points_rebounds_assists...
Saved 28 players for player_points_rebounds_assists


,NAME,LINE,L-5,L-10,L-15
0,Saddiq Bey,25.5,0.8,0.7,0.53
1,Jay Huff,18.5,0.8,0.5,0.40
2,Precious Achiuwa,12.5,0.8,0.6,0.53
3,Julian Champagnie,14.5,0.8,0.7,0.67
4,Keldon Johnson,20.5,0.8,0.9,0.67
5,Donte DiVincenzo,21.5,0.8,0.6,0.47
6,Devin Vassell,21.5,0.8,0.8,0.53
7,Royce O'Neale,18.5,0.6,0.4,0.53
8,Naz Reid,23.5,0.6,0.5,0.47
9,Jordan Hawkins,9.5,0.6,0.5,0.40



Processing player_points_rebounds...
Saved 18 players for player_points_rebounds


,NAME,LINE,L-5,L-10,L-15
0,Saddiq Bey,22.5,1.0,0.8,0.60
1,Pascal Siakam,32.5,0.8,0.6,0.47
2,Jeremiah Fears,20.5,0.8,0.5,0.53
3,Stephon Castle,18.5,0.8,0.9,0.80
4,Devin Vassell,19.5,0.8,0.7,0.47
5,Naz Reid,21.5,0.6,0.6,0.53
6,Trey Murphy III,27.5,0.6,0.6,0.60
7,Collin Gillespie,21.5,0.6,0.6,0.40
8,Mark Williams,20.5,0.6,0.5,0.53
9,Jaden McDaniels,20.5,0.6,0.3,0.40



Processing player_points_assists...
Saved 11 players for player_points_assists


,NAME,LINE,L-5,L-10,L-15
0,Stephon Castle,18.5,0.8,0.9,0.73
1,Anthony Edwards,33.5,0.6,0.6,0.60
2,Julius Randle,28.5,0.6,0.4,0.53
3,Dillon Brooks,24.5,0.6,0.6,0.47
4,Collin Gillespie,23.5,0.6,0.5,0.33
5,De'Aaron Fox,30.5,0.6,0.6,0.47
6,Trey Murphy III,25.5,0.6,0.4,0.47
7,Grayson Allen,22.5,0.4,0.2,0.27
8,Pascal Siakam,28.5,0.2,0.4,0.47
9,DeMar DeRozan,23.5,0.2,0.3,0.27



Processing player_rebounds_assists...
Saved 6 players for player_rebounds_assists


,NAME,LINE,L-5,L-10,L-15
0,Maxime Raynaud,8.5,0.6,0.3,0.27
1,Saddiq Bey,8.5,0.6,0.6,0.53
2,Pascal Siakam,11.5,0.4,0.3,0.27
3,T.J. McConnell,7.5,0.4,0.4,0.27
4,Anthony Edwards,9.5,0.4,0.4,0.40
5,Rudy Gobert,12.5,0.4,0.7,0.60



Processing player_turnovers...
No data found for player_turnovers

Processing player_blocks_steals...
No data found for player_blocks_steals
